In [0]:
# Databricks notebook source
# MAGIC %md # Nifty 50 — Data Quality Check
 
# COMMAND ----------
 
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from datetime import date
 
# ── Config ────────────────────────────────────────────────────────────────
SESSION_DATE   = str(date.today())   # change to "2024-01-15" for backfill
DELTA_TABLE    = "nifty_live.session_ticks"
EXPECTED_TICKS = 375                 # 9:15 to 15:30, one per minute
ZSCORE_THRESH  = 3.5
 
print(f"Session : {SESSION_DATE}")
print(f"Table   : {DELTA_TABLE}")
 
# COMMAND ----------
# MAGIC %md ## Load Data
 
# COMMAND ----------
 
df  = spark.table(DELTA_TABLE).filter(F.col("date") == SESSION_DATE)
cnt = df.count()
 
if cnt == 0:
    print(f"NO DATA found for {SESSION_DATE} — pipeline may not have run.")
    dbutils.notebook.exit("NO DATA")
 
pdf = df.orderBy("datetime_ist").toPandas()
pdf["datetime_ist"] = pd.to_datetime(pdf["datetime_ist"])
 
print(f"Loaded {cnt} ticks for {SESSION_DATE}")
 
# COMMAND ----------
# MAGIC %md ## Check 1 — Tick Count & Gaps
 
# COMMAND ----------
 
missing = EXPECTED_TICKS - cnt
pdf["gap_sec"] = pdf["datetime_ist"].diff().dt.total_seconds()
 
print("=" * 50)
print("TICK COUNT & GAPS")
print("=" * 50)
print(f"Expected ticks     : {EXPECTED_TICKS}")
print(f"Actual ticks       : {cnt}")
print(f"Missing ticks      : {missing}  {'✓' if missing <= 5 else '⚠ WARNING' if missing <= 20 else '✗ CRITICAL'}")
print(f"Max gap (seconds)  : {pdf['gap_sec'].max():.0f}")
print(f"Gaps > 90s         : {(pdf['gap_sec'] > 90).sum()}")
print(f"Gaps > 300s        : {(pdf['gap_sec'] > 300).sum()}")
print(f"First tick         : {pdf['datetime_ist'].iloc[0]}")
print(f"Last tick          : {pdf['datetime_ist'].iloc[-1]}")
 
# COMMAND ----------
# MAGIC %md ## Check 2 — OHLC Integrity
 
# COMMAND ----------
 
high_lt_low    = int((pdf["high"] < pdf["low"]).sum())
close_gt_high  = int((pdf["close"] > pdf["high"]).sum())
close_lt_low   = int((pdf["close"] < pdf["low"]).sum())
open_gt_high   = int((pdf["open"] > pdf["high"]).sum())
open_lt_low    = int((pdf["open"] < pdf["low"]).sum())
null_count     = int(pdf[["open","high","low","close"]].isnull().sum().sum())
zero_price     = int((pdf["close"] <= 0).sum())
total_violations = high_lt_low + close_gt_high + close_lt_low + open_gt_high + open_lt_low + null_count + zero_price
 
print("=" * 50)
print("OHLC INTEGRITY")
print("=" * 50)
print(f"high < low         : {high_lt_low}   {'✓' if high_lt_low == 0 else '✗ CRITICAL'}")
print(f"close > high       : {close_gt_high}   {'✓' if close_gt_high == 0 else '✗ CRITICAL'}")
print(f"close < low        : {close_lt_low}   {'✓' if close_lt_low == 0 else '✗ CRITICAL'}")
print(f"open > high        : {open_gt_high}   {'✓' if open_gt_high == 0 else '✗ CRITICAL'}")
print(f"open < low         : {open_lt_low}   {'✓' if open_lt_low == 0 else '✗ CRITICAL'}")
print(f"null prices        : {null_count}   {'✓' if null_count == 0 else '✗ CRITICAL'}")
print(f"zero/neg close     : {zero_price}   {'✓' if zero_price == 0 else '✗ CRITICAL'}")
print(f"─" * 35)
print(f"Total violations   : {total_violations}   {'✓ CLEAN' if total_violations == 0 else '✗ ISSUES FOUND'}")
 
# COMMAND ----------
# MAGIC %md ## Check 3 — Session Summary
 
# COMMAND ----------
 
s_open  = float(pdf["open"].iloc[0])
s_close = float(pdf["close"].iloc[-1])
s_high  = float(pdf["high"].max())
s_low   = float(pdf["low"].min())
s_range = s_high - s_low
s_range_pct  = round(s_range / s_open * 100, 3)
s_return_pct = round((s_close - s_open) / s_open * 100, 3)
 
print("=" * 50)
print("SESSION SUMMARY")
print("=" * 50)
print(f"Open               : {s_open:.2f}")
print(f"Close              : {s_close:.2f}")
print(f"High               : {s_high:.2f}")
print(f"Low                : {s_low:.2f}")
print(f"Intraday range     : {s_range:.2f}  ({s_range_pct}%)")
print(f"Session return     : {s_return_pct}%  {'▲' if s_return_pct >= 0 else '▼'}")
print(f"Is expiry day      : {'Yes ✓' if pdf['expiry'].max() == 1 else 'No'}")
print(f"Pre-market ticks   : {int(pdf['pre_market'].sum())}")
print(f"Post-market ticks  : {int(pdf['post_market'].sum())}")
 
if s_range_pct < 0.10:
    print(f"⚠ WARNING: Very flat session — verify data source")
if s_range_pct > 8.0:
    print(f"ℹ INFO: Wide session — possible circuit breaker day")
 
# COMMAND ----------
# MAGIC %md ## Check 4 — Anomaly Detection
 
# COMMAND ----------
 
close   = pdf["close"].values
mu      = np.mean(close)
sigma   = np.std(close)
zscores = (close - mu) / sigma if sigma > 0 else np.zeros(len(close))
pdf["z_score"] = zscores
 
anomalies = pdf[np.abs(zscores) > ZSCORE_THRESH][["time_ist","open","high","low","close","z_score"]]
 
# IQR
q1, q3 = np.percentile(close, 25), np.percentile(close, 75)
iqr     = q3 - q1
iqr_outliers = int(((close < q1 - 2.5*iqr) | (close > q3 + 2.5*iqr)).sum())
 
# Single-tick extreme moves
pdf["tick_return_pct"] = pdf["close"].pct_change() * 100
extreme_moves = int((pdf["tick_return_pct"].abs() > 3.0).sum())
 
print("=" * 50)
print("ANOMALY DETECTION")
print("=" * 50)
print(f"Mean close         : {mu:.2f}")
print(f"Std close          : {sigma:.2f}")
print(f"Z-score anomalies  : {len(anomalies)}  (|z| > {ZSCORE_THRESH})")
print(f"IQR outliers       : {iqr_outliers}")
print(f"Extreme tick moves : {extreme_moves}  (>3% in one minute)")
 
if len(anomalies) > 0:
    print(f"\nAnomalous ticks:")
    print(anomalies.to_string(index=False))
 
# COMMAND ----------
# MAGIC %md ## Check 5 — Volatility
 
# COMMAND ----------
 
pdf["log_ret"] = np.log(pdf["close"] / pdf["close"].shift(1))
ann_factor     = np.sqrt(252 * 375)
 
realised_vol   = round(float(pdf["log_ret"].std() * ann_factor), 4)
parkinson_vol  = round(float(
    np.sqrt((1/(4*np.log(2))) * np.mean((np.log(pdf["high"]/pdf["low"])**2))) * ann_factor
), 4)
vol_5m  = round(float(pdf["log_ret"].rolling(5).std().mean()  * ann_factor), 4)
vol_15m = round(float(pdf["log_ret"].rolling(15).std().mean() * ann_factor), 4)
vol_60m = round(float(pdf["log_ret"].rolling(60).std().mean() * ann_factor), 4)
 
print("=" * 50)
print("VOLATILITY  (annualised)")
print("=" * 50)
print(f"Realised vol       : {realised_vol}")
print(f"Parkinson vol      : {parkinson_vol}")
print(f"Avg rolling 5m     : {vol_5m}")
print(f"Avg rolling 15m    : {vol_15m}")
print(f"Avg rolling 60m    : {vol_60m}")
print(f"Note: save these — after 20 sessions you have a vol baseline for anomaly thresholds")
 
# COMMAND ----------
# MAGIC %md ## Final Verdict
 
# COMMAND ----------
 
critical = []
warnings = []
 
if missing > 20:            critical.append(f"Missing {missing} ticks")
elif missing > 5:           warnings.append(f"Missing {missing} ticks")
if total_violations > 0:    critical.append(f"{total_violations} OHLC violations")
if null_count > 0:          critical.append(f"{null_count} null prices")
if (pdf["gap_sec"] > 300).sum() > 0:
                            critical.append(f"{int((pdf['gap_sec']>300).sum())} gaps >5 min")
if s_range_pct < 0.10:     critical.append(f"Suspiciously flat session ({s_range_pct}%)")
if len(anomalies) > 10:    warnings.append(f"{len(anomalies)} anomalous ticks")
if extreme_moves > 5:      warnings.append(f"{extreme_moves} extreme single-tick moves")
 
status = "✗ CRITICAL" if critical else "⚠ WARNING" if warnings else "✓ PASS"
 
print("=" * 50)
print(f"FINAL STATUS : {status}")
print("=" * 50)
for c in critical: print(f"  ✗ {c}")
for w in warnings: print(f"  ⚠ {w}")
if not critical and not warnings:
    print("  All checks passed. Data looks clean.")